In [1]:
%pip install dash plotly pandas
%pip install -U nbformat ipykernel plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#import packages
import pandas as pd
import plotly.express as px
from dash import Dash, dcc, html, dash_table, Input, Output
import nbformat
import plotly


In [ ]:
#import datasets
customers = pd.read_csv("data/MavenMarket_Customers.csv")
products = pd.read_csv("data/MavenMarket_Products.csv")
regions = pd.read_csv("data/MavenMarket_Regions.csv")
stores = pd.read_csv("data/MavenMarket_Stores.csv")
trans_1997 = pd.read_csv("data/MavenMarket_Transactions_1997.csv")
trans_1998 = pd.read_csv("data/MavenMarket_Transactions_1998.csv")

In [ ]:
#data preprocessing
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^\w]", "", regex=True)
    )
    return df

customers = clean_columns(customers)
products = clean_columns(products)
regions = clean_columns(regions)
stores = clean_columns(stores)
trans_1997 = clean_columns(trans_1997)
trans_1998 = clean_columns(trans_1998)

In [ ]:
#data transformation and aggregation
transactions = pd.concat([trans_1997, trans_1998], ignore_index=True)

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"], errors="coerce"
)

transactions["year"] = transactions["transaction_date"].dt.year
transactions["month"] = transactions["transaction_date"].dt.month
transactions["month_name"] = transactions["transaction_date"].dt.strftime("%b")
transactions["quarter"] = transactions["transaction_date"].dt.quarter

for df_obj, col in [
    (customers, "customer_id"),
    (products, "product_id"),
    (regions, "region_id"),
    (stores, "store_id"),
    (stores, "region_id"),
    (transactions, "customer_id"),
    (transactions, "product_id"),
    (transactions, "store_id"),
]:
    if col in df_obj.columns:
        df_obj[col] = df_obj[col].astype(str)

#joining dataset on keys 
stores_regions = stores.merge(regions, on="region_id", how="left")

df = transactions.merge(customers, on="customer_id", how="left")
df = df.merge(products, on="product_id", how="left")
df = df.merge(stores_regions, on="store_id", how="left")

#define main metrics
df["sales"] = df["quantity"] * df["product_retail_price"]
df["cost"] = df["quantity"] * df["product_cost"]
df["profit"] = df["sales"] - df["cost"]

df.head()

,transaction_date,stock_date,product_id,customer_id,store_id,quantity,year,month,month_name,quarter,...,store_phone,first_opened_date,last_remodel_date,total_sqft,grocery_sqft,sales_district,sales_region,sales,cost,profit
0,1997-01-01,12/31/1996,869,3449,6,5,1997,1,Jan,1,...,958-555-5002,1/3/1981,3/13/1991,23688,15337,Los Angeles,South West,10.60,4.55,6.05
1,1997-01-01,12/31/1996,1472,3449,6,3,1997,1,Jan,1,...,958-555-5002,1/3/1981,3/13/1991,23688,15337,Los Angeles,South West,6.60,2.70,3.90
2,1997-01-01,12/28/1996,76,3449,6,4,1997,1,Jan,1,...,958-555-5002,1/3/1981,3/13/1991,23688,15337,Los Angeles,South West,6.76,2.76,4.00
3,1997-01-01,12/26/1996,320,3449,6,3,1997,1,Jan,1,...,958-555-5002,1/3/1981,3/13/1991,23688,15337,Los Angeles,South West,9.78,3.24,6.54
4,1997-01-01,12/25/1996,4,3449,6,4,1997,1,Jan,1,...,958-555-5002,1/3/1981,3/13/1991,23688,15337,Los Angeles,South West,14.56,6.56,8.00


In [ ]:
#shape and columns checking
print(df.columns.tolist())
print(df.shape)

In [ ]:
#define filters attributes
def apply_filters(df, years=None, regions=None, genders=None, educations=None, occupations=None):
    filtered = df.copy()

    if years:
        filtered = filtered[filtered["year"].isin(years)]

    if regions and "sales_region" in filtered.columns:
        filtered = filtered[filtered["sales_region"].isin(regions)]

    if genders and "gender" in filtered.columns:
        filtered = filtered[filtered["gender"].isin(genders)]

    if educations and "education" in filtered.columns:
        filtered = filtered[filtered["education"].isin(educations)]

    if occupations and "occupation" in filtered.columns:
        filtered = filtered[filtered["occupation"].isin(occupations)]

    return filtered

#kpi cards
def kpi_summary(df):
    return {
        "Total Sales": f"${df['sales'].sum():,.2f}",
        "Total Profit": f"${df['profit'].sum():,.2f}",
        "Total Quantity": f"{df['quantity'].sum():,.0f}",
        "Active Customers": f"{df['customer_id'].nunique():,}",
        "Active Stores": f"{df['store_id'].nunique():,}",
    }

#top product table
def top_products_table(df):
    result = (
        df.groupby(["product_name", "product_brand"], as_index=False)[["sales", "profit", "quantity"]]
        .sum()
        .sort_values("sales", ascending=False)
        .head(15)
    )
    result["sales"] = result["sales"].round(2)
    result["profit"] = result["profit"].round(2)
    return result

In [ ]:
#create dashapp
app = Dash(__name__)

years = sorted(df["year"].dropna().astype(int).unique().tolist())
regions_list = sorted(df["sales_region"].dropna().astype(str).unique().tolist()) if "sales_region" in df.columns else []
genders = sorted(df["gender"].dropna().astype(str).unique().tolist()) if "gender" in df.columns else []
educations = sorted(df["education"].dropna().astype(str).unique().tolist()) if "education" in df.columns else []
occupations = sorted(df["occupation"].dropna().astype(str).unique().tolist()) if "occupation" in df.columns else []

app.layout = html.Div([
    html.H1("Maven Market Dashboard"),

    html.Div([
        dcc.Dropdown(id="year_filter",
                     options=[{"label": str(y), "value": y} for y in years],
                     value=years, multi=True, placeholder="Select year"),
        dcc.Dropdown(id="region_filter",
                     options=[{"label": r, "value": r} for r in regions_list],
                     value=[], multi=True, placeholder="Select region"),
        dcc.Dropdown(id="gender_filter",
                     options=[{"label": g, "value": g} for g in genders],
                     value=[], multi=True, placeholder="Select gender"),
        dcc.Dropdown(id="education_filter",
                     options=[{"label": e, "value": e} for e in educations],
                     value=[], multi=True, placeholder="Select education"),
        dcc.Dropdown(id="occupation_filter",
                     options=[{"label": o, "value": o} for o in occupations],
                     value=[], multi=True, placeholder="Select occupation"),
    ], style={"display": "grid", "gridTemplateColumns": "1fr 1fr 1fr 1fr 1fr", "gap": "10px"}),

    html.Br(),
    html.Div(id="kpi_cards", style={"display": "flex", "gap": "10px"}),

    html.Div([
        dcc.Graph(id="monthly_sales_chart"),
        dcc.Graph(id="sales_by_region_chart"),
        dcc.Graph(id="profit_by_brand_chart"),
        dcc.Graph(id="member_card_chart"),
        dcc.Graph(id="sales_by_education_chart"),
        dcc.Graph(id="sales_by_occupation_chart"),
        dcc.Graph(id="top_store_sales_chart"),
        dcc.Graph(id="sales_by_store_type_chart"),
    ], style={"display": "grid", "gridTemplateColumns": "1fr 1fr", "gap": "20px"}),

    html.H3("Top 15 Products"),
    dash_table.DataTable(
        id="top_products_table",
        page_size=15,
        style_table={"overflowX": "auto"},
        style_cell={"textAlign": "left"}
    )
])

In [ ]:
#defind callback layout
@app.callback(
    Output("kpi_cards", "children"),
    Output("monthly_sales_chart", "figure"),
    Output("sales_by_region_chart", "figure"),
    Output("profit_by_brand_chart", "figure"),
    Output("member_card_chart", "figure"),
    Output("top_products_table", "data"),
    Output("top_products_table", "columns"),
    Output("sales_by_education_chart", "figure"),
    Output("sales_by_occupation_chart", "figure"),
    Output("top_store_sales_chart", "figure"),
    Output("sales_by_store_type_chart", "figure"),
    Input("year_filter", "value"),
    Input("region_filter", "value"),
    Input("gender_filter", "value"),
    Input("education_filter", "value"),
    Input("occupation_filter", "value"),
)
def update_dashboard(year_values, region_values, gender_values, education_values, occupation_values):
    filtered = apply_filters(
        df,
        years=year_values,
        regions=region_values,
        genders=gender_values,
        educations=education_values,
        occupations=occupation_values,
    )

    if filtered.empty:
        empty_fig = px.bar(title="No data for selected filters")
        return (
            [],
            empty_fig,
            empty_fig,
            empty_fig,
            empty_fig,
            [],
            [],
            empty_fig,
            empty_fig,
            empty_fig,
            empty_fig
        )

    kpis = kpi_summary(filtered)
    cards = [
        html.Div([
            html.H4(k),
            html.P(v)
        ], style={
            "border": "1px solid #ddd",
            "padding": "12px",
            "borderRadius": "8px",
            "flex": "1"
        })
        for k, v in kpis.items()
    ]

    month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                   "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

    monthly = filtered.groupby(["year", "month_name"], as_index=False)["sales"].sum()
    monthly["month_name"] = pd.Categorical(monthly["month_name"], categories=month_order, ordered=True)
    monthly = monthly.sort_values(["year", "month_name"])
    fig_monthly = px.line(monthly, x="month_name", y="sales", color="year", markers=True, title="Monthly Sales Trend")

    by_region = filtered.groupby("sales_region", as_index=False)["sales"].sum()
    fig_region = px.bar(by_region, x="sales_region", y="sales", title="Sales by Region")

    by_brand = (
        filtered.groupby("product_brand", as_index=False)["profit"]
        .sum()
        .sort_values("profit", ascending=False)
        .head(10)
    )
    fig_brand = px.bar(by_brand, x="product_brand", y="profit", title="Top 10 Brands by Profit")

    member = filtered.groupby("member_card", as_index=False)["customer_id"].nunique()
    fig_member = px.pie(member, names="member_card", values="customer_id", title="Customers by Member Card")

    by_store = (
    filtered.groupby("store_name", as_index=False)["sales"]
    .sum()
    .sort_values("sales", ascending=False)
    .head(10)
)

    fig_store = px.bar(by_store,x="store_name",y="sales",title="Top 10 Stores by Sales"
)

    by_education = (
        filtered.groupby("education", as_index=False)["sales"]
        .sum()
        .sort_values("sales", ascending=False)
    )
    fig_education = px.bar(by_education, x="education", y="sales", title="Sales by Education")

    by_occupation = (
        filtered.groupby("occupation", as_index=False)["sales"]
        .sum()
        .sort_values("sales", ascending=False)
        .head(10)
    )
    fig_occupation = px.bar(by_occupation, x="occupation", y="sales", title="Top 10 Occupations by Sales")

    by_store_type = (
        filtered.groupby("store_type", as_index=False)["sales"]
        .sum()
        .sort_values("sales", ascending=False)
    )
    fig_store_type = px.bar(by_store_type, x="store_type", y="sales", title="Sales by Store Type")

    table_df = top_products_table(filtered)
    table_data = table_df.to_dict("records")
    table_columns = [{"name": c.replace("_", " ").title(), "id": c} for c in table_df.columns]

    return (
    cards,
    fig_monthly,
    fig_region,
    fig_brand,
    fig_member,
    table_data,
    table_columns,
    fig_store,
    fig_education,
    fig_occupation,
    fig_store_type
)

In [ ]:
#run dashapp on web browser
import webbrowser
from threading import Timer

def open_browser():
    webbrowser.open_new("http://127.0.0.1:8060/")

if __name__ == "__main__":
    Timer(1, open_browser).start()
    app.run(debug=True, port=8060)